# SPIN-Soft — Step 3/4: generation (baseline + soft-SPIN answers), **swept over α** (master, all non-Gemma models)

Generates SALAD fairness/privacy answers for the **baseline** model and the **soft-SPIN** model at each α
(coupled weights rescaled to αW). Greedy decoding, `max_new_tokens=64`, on the **raw** question — matching
`eval_salad.generate_answers` (HuggingFace greedy in place of the authors' vLLM; identical greedy outputs).

Same internal-α pattern as Step 2: model loaded once, original coupled weights cached once, each α resets to
αW before generating. α=0 = hard SPIN, α=1 = baseline (soft column ≡ baseline column — a free check).

**Checkpointed per (α, set)** to `gen_ckpt_<tag>.json` → a Kaggle timeout resumes at the exact batch; nothing
regenerated. **Input:** this model's `coupled_soft_<tag>.json`. **Output:** `answers_<tag>_a{α}_{fairness,privacy}.csv`
+ `results_soft_gen_<tag>.json` → score in Step 4. GPU **T4 x2**, Internet ON.

> **Gemma-2 uses a dedicated notebook** (`SPIN_soft_step3_generate_gemma.ipynb`): generation needs the KV cache,
> so Gemma is loaded on a **single T4** (cache on, no cross-device) rather than cache-disabled. This master is for
> Llama-3.2-3B, Qwen2.5-3B and the 7B, which tolerate the split during generation.

## Step 0 — env + config + **model select** + find coupled set

In [1]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import subprocess, sys, glob, json, gc, time
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.46.3','sentencepiece'], check=False)
import pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ============================================================
# SELECT MODEL — uncomment EXACTLY ONE. (Gemma has its own notebook.)
# ------------------------------------------------------------
# MODEL_ID,USE_FAST = 'meta-llama/Llama-3.2-3B-Instruct', True   # POC (gated; fast tokenizer)
#MODEL_ID,USE_FAST = 'Qwen/Qwen2.5-3B-Instruct',        False   # POC
MODEL_ID,USE_FAST = 'Qwen/Qwen2-7B-Instruct',          False   # scale-up
# ============================================================
ALPHAS=[0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]   # locked grid; 0=hard SPIN, 1=baseline
# GENERATION IS THE EXPENSIVE STEP. Levers if GPU-bound:
#   - N_EVAL: cap SALAD questions per axis (None = full). Set an int (e.g. 250) for the 7B.
#   - ALPHAS: α=1 duplicates baseline (soft==baseline); drop it and/or 0.1 on the 7B to save passes.
N_EVAL=None
USE_CHAT_TEMPLATE=False       # authors feed RAW questions (verified)
GEN_BS=4
MAXNEW=64                     # eval_salad SamplingParams(max_tokens=64)
DTYPE=torch.float16
MODEL_TAG=MODEL_ID.split('/')[-1]

def find(name):
    for r in ['.','/kaggle/input','/kaggle/working']:
        h=glob.glob(os.path.join(r,'**',name),recursive=True)
        if h: return h[0]
    return None
CP=find(f'coupled_soft_{MODEL_TAG}.json'); assert CP, f'add coupled_soft_{MODEL_TAG}.json (Step-1 output) as input'
meta=json.load(open(CP)); COUPLED=meta['coupled']
NZ=sum(len(v) for v in COUPLED.values())
print('SoftSPIN Step 3 |', MODEL_TAG, '| coupled set:', CP, '| coupled weights', NZ, '| alphas', ALPHAS)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 96.9 MB/s eta 0:00:00
SoftSPIN Step 3 | Qwen2-7B-Instruct | coupled set: /kaggle/input/notebooks/testerzone/softspin-step-1-qwen-7b/coupled_soft_Qwen2-7B-Instruct.json | coupled weights 1148 | alphas [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]


## Step 1 — data + model + **cache original coupled weights**

In [2]:
DEAN_DIR='/tmp/DEAN'   # clone to scratch, NOT /kaggle/working, so DEAN is never saved to notebook output
if not os.path.exists(DEAN_DIR):
    subprocess.run(['git','clone','--depth','1','https://github.com/AI45Lab/DEAN.git',DEAN_DIR], check=True)
DATA=f'{DEAN_DIR}/src/data'
fair_q=pd.read_csv(f'{DATA}/salad_fairness.csv')['questions'].tolist()[:N_EVAL]
priv_q=pd.read_csv(f'{DATA}/salad_privacy.csv')['questions'].tolist()[:N_EVAL]

tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=USE_FAST, trust_remote_code=True)
tokenizer.padding_side='left'
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True,
        device_map='auto', max_memory={0:'14GiB',1:'14GiB','cpu':'28GiB'}).eval()   # split across T4x2
GEN_DEV=model.get_input_embeddings().weight.device
print('model loaded | entry', GEN_DEV, '|', len(fair_q), 'fair +', len(priv_q), 'priv questions')

# ---- cache ORIGINAL coupled-weight values ONCE (untouched = baseline) ----
PARAMS=dict(model.named_parameters()); IDX={}; ORIG={}
for n,idxlist in COUPLED.items():
    if n in PARAMS and idxlist:
        p=PARAMS[n]; idx=torch.tensor(idxlist, device=p.device)
        IDX[n]=idx; ORIG[n]=p.data.view(-1)[idx].clone()
print('cached originals for', len(IDX), 'matrices,', sum(v.numel() for v in IDX.values()), 'weights')

Cloning into '/tmp/DEAN'...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

model loaded | entry cuda:0 | 2165 fair + 657 priv questions
cached originals for 84 matrices, 1148 weights


## Step 2 — functions (raw-question format, reset+scale-to-αW, checkpointed greedy generation)

In [3]:
def fmt(chunk):
    if USE_CHAT_TEMPLATE:
        return [tokenizer.apply_chat_template([{'role':'user','content':q}],tokenize=False,add_generation_prompt=True) for q in chunk]
    return [str(q) for q in chunk]
@torch.no_grad()
def apply_alpha(alpha):
    nz=0
    for n,idx in IDX.items():
        PARAMS[n].data.view(-1)[idx]=alpha*ORIG[n]; nz+=idx.numel()
    return nz
CK=f'/kaggle/working/gen_ckpt_{MODEL_TAG}.json'
state=json.load(open(CK)) if os.path.exists(CK) else {}
@torch.no_grad()
def gen_set(key, queries):
    done=state.get(key, []); t0=time.time()
    for i in range(len(done), len(queries), GEN_BS):
        enc=tokenizer(fmt(queries[i:i+GEN_BS]),return_tensors='pt',padding=True,truncation=True,max_length=256).to(GEN_DEV)
        g=model.generate(**enc,max_new_tokens=MAXNEW,do_sample=False,pad_token_id=tokenizer.pad_token_id)
        done+=[tokenizer.decode(g[j,enc.input_ids.shape[1]:],skip_special_tokens=True).strip() for j in range(g.shape[0])]
        state[key]=done; json.dump(state, open(CK,'w'))
        print(f'    {key} {len(done)}/{len(queries)} ({time.time()-t0:.0f}s)', end='\r')
    print(); return done
print('functions ready.')

functions ready.


## Step 3 — baseline once, then generate soft answers per α (checkpointed, resumable)

In [4]:
# baseline (untouched weights) — generated once, reused as the comparison column in every alpha file
print('baseline generation (untouched weights)...')
base_fair=gen_set('base_fair', fair_q); base_priv=gen_set('base_priv', priv_q)

tag=MODEL_TAG
for a in ALPHAS:
    key=f'{a:.2f}'
    nz=apply_alpha(a)   # coupled <- alpha*orig  (absolute; safe on resume since orig is the untouched snapshot)
    print(f'alpha={key}: set {nz} coupled weights -> generating...')
    sf=gen_set(f'a{key}_fair', fair_q); sp=gen_set(f'a{key}_priv', priv_q)
    pd.DataFrame({'q':fair_q,'baseline':base_fair,'soft':sf}).to_csv(f'/kaggle/working/answers_{tag}_a{key}_fairness.csv',index=False)
    pd.DataFrame({'q':priv_q,'baseline':base_priv,'soft':sp}).to_csv(f'/kaggle/working/answers_{tag}_a{key}_privacy.csv',index=False)
    print(f'  saved answers_{tag}_a{key}_*.csv')

json.dump({'model':MODEL_ID,'tag':tag,'variant':'soft-SPIN','coupled_count':NZ,
           'alphas':ALPHAS,'n_eval':N_EVAL,'n_fair':len(fair_q),'n_priv':len(priv_q)},
          open(f'/kaggle/working/results_soft_gen_{tag}.json','w'), indent=2)
print(f'\nSTEP 3 DONE ({tag}). Per-alpha answer CSVs + results_soft_gen_{tag}.json saved.')
print('Next: Step 4 (MD-Judge) scores every answers_*_a{alpha}_*.csv -> baseline-vs-soft safe-rate vs alpha.')

baseline generation (untouched weights)...


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


    base_fair 2165/2165 (2405s)
    base_priv 657/657 (734s)
alpha=0.00: set 1148 coupled weights -> generating...
    a0.00_fair 2165/2165 (2411s)
    a0.00_priv 657/657 (734s)
  saved answers_Qwen2-7B-Instruct_a0.00_*.csv
alpha=0.10: set 1148 coupled weights -> generating...
    a0.10_fair 2165/2165 (2413s)
    a0.10_priv 657/657 (735s)
  saved answers_Qwen2-7B-Instruct_a0.10_*.csv
alpha=0.30: set 1148 coupled weights -> generating...
    a0.30_fair 2165/2165 (2418s)
    a0.30_priv 657/657 (736s)
  saved answers_Qwen2-7B-Instruct_a0.30_*.csv
alpha=0.50: set 1148 coupled weights -> generating...
    a0.50_fair 2165/2165 (2422s)
    a0.50_priv 657/657 (738s)
  saved answers_Qwen2-7B-Instruct_a0.50_*.csv
alpha=0.70: set 1148 coupled weights -> generating...
    a0.70_fair 2165/2165 (2424s)
    a0.70_priv 657/657 (738s)
  saved answers_Qwen2-7B-Instruct_a0.70_*.csv
alpha=0.90: set 1148 coupled weights -> generating...
    a0.90_fair 2165/2165 (2428s)
    a0.90_priv 657/657 (738s)
  saved